## 1. Install Dependencies

In [ ]:
!pip install torch transformers datasets peft accelerate bitsandbytes

## 2. Prepare Training Data

Create training examples from your workflow executions

In [ ]:
import json

# Example training data for workflow planning
training_data = [
    {
        "instruction": "Plan a client onboarding workflow",
        "input": "Client: Acme Corp, needs Jira project and Stripe billing",
        "output": json.dumps([
            {"step": 1, "action": "Create Jira project", "tool": "jira_create_project"},
            {"step": 2, "action": "Create Stripe customer", "tool": "stripe_create_customer"},
            {"step": 3, "action": "Setup subscription", "tool": "stripe_create_subscription"},
            {"step": 4, "action": "Send welcome email", "tool": "send_email"}
        ])
    },
    {
        "instruction": "Validate workflow execution result",
        "input": "Step: Create Jira project, Result: Error - Invalid key format",
        "output": json.dumps({
            "valid": False,
            "errors": ["Invalid project key format"],
            "suggestions": ["Remove special characters from project key"]
        })
    },
    {
        "instruction": "Correct workflow step parameters",
        "input": "Error: Invalid project key 'TEST-123', Suggestion: Remove hyphens",
        "output": json.dumps({
            "corrected_params": {"project_key": "TEST123"},
            "reasoning": "Removed hyphens to match Jira key format requirements"
        })
    }
]

# Save training data
with open('workflow_training_data.json', 'w') as f:
    json.dump(training_data, f, indent=2)

print(f"Created {len(training_data)} training examples")

## 3. Load Base Model (Local)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Use a smaller model that can run locally
model_name = "microsoft/phi-2"  # 2.7B parameters - runs on 8GB RAM
# Alternative: "TinyLlama/TinyLlama-1.1B-Chat-v1.0" for even lower memory

print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Model loaded successfully!")

## 4. Prepare Dataset

In [ ]:
from datasets import Dataset

def format_prompt(example):
    """Format training example as prompt"""
    return {
        "text": f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Output:
{example['output']}"""
    }

# Create dataset
dataset = Dataset.from_list(training_data)
dataset = dataset.map(format_prompt)

print(f"Dataset size: {len(dataset)}")
print("\nExample:")
print(dataset[0]['text'])

## 5. Configure LoRA Fine-tuning

Using LoRA (Low-Rank Adaptation) for efficient fine-tuning

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# LoRA configuration
lora_config = LoraConfig(
    r=8,  # Rank
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],  # Which layers to adapt
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Prepare model for training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

## 6. Train the Model

In [ ]:
from transformers import TrainingArguments, Trainer

# Training configuration
training_args = TrainingArguments(
    output_dir="./workflow-model-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    save_steps=10,
    logging_steps=10,
    save_total_limit=2,
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

# Start training
print("Starting training...")
trainer.train()
print("Training complete!")

## 7. Save Fine-tuned Model

In [ ]:
# Save the fine-tuned model
model.save_pretrained("./workflow-model-finetuned")
tokenizer.save_pretrained("./workflow-model-finetuned")

print("Model saved to ./workflow-model-finetuned")

## 8. Test Fine-tuned Model

In [ ]:
def test_model(prompt):
    """Test the fine-tuned model"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test workflow planning
test_prompt = """### Instruction:
Plan a client onboarding workflow

### Input:
Client: TechCorp, needs Jira project, Stripe billing, and welcome email

### Output:"""

result = test_model(test_prompt)
print(result)

## 9. Convert to Ollama Format (Optional)

In [ ]:
# Create Modelfile for Ollama
modelfile = """
FROM ./workflow-model-finetuned

PARAMETER temperature 0.7
PARAMETER top_p 0.9

SYSTEM You are a workflow automation agent specialized in planning and executing business processes.
"""

with open('Modelfile', 'w') as f:
    f.write(modelfile)

print("Created Modelfile")
print("\nTo use with Ollama:")
print("1. ollama create workflow-agent -f Modelfile")
print("2. Update OLLAMA_MODEL=workflow-agent in .env")

## 10. Model Comparison

In [ ]:
# Compare base model vs fine-tuned model
test_cases = [
    "Plan a client onboarding workflow",
    "Validate a Jira project creation result",
    "Correct an invalid project key parameter"
]

print("Testing fine-tuned model on workflow tasks...\n")
for test in test_cases:
    prompt = f"""### Instruction:
{test}

### Output:"""
    result = test_model(prompt)
    print(f"Task: {test}")
    print(f"Result: {result[:200]}...\n")